Urban Data Science & Smart Cities <br>
URSP688Y Spring 2026<br>
Instructor: Chester Harvey <br>
Urban Studies & Planning <br>
National Center for Smart Growth <br>
University of Maryland

# Exercise04

This last exercise is an opportunity for you to get started on your final project. Please identify a portion of your project to get started on and submit a notebook (and any other related files) where you:

1. State the question you are aiming to address with this portion of your analysis
2. Outline the approach you will use to answer that question (pseudocode or you can start to more formally outline the approach section for your final narrative)
3. Operationalize your approach with data and code that you can later slot into your final analysis

## Submitting

Please make a pull request with all of your code and reasonably-sized data in a folder named with your first name. See my example file structure in the `exercise04` directory:

```
exercises
└── exercise04
    └── chester
        ├── example_ReadMe.md
        ├── exercise04_chester.ipynb
        ├── example_module.py
        └── example_data.csv
```

If you have datasets that are too large for GitHub or should not be made public, please upload them to the [Google Drive folder for final project data](https://drive.google.com/drive/folders/1jgLIxx1B67nD4PekfDJnUI5jmrqa4iZm). Please also provide instructions for how someone running your code should properly locate or connect to these files so the analysis will run properly. For example, should they copy and paste the files into the same directory as your notebook, or a provided `data` directory? Best practice is to include these instructions in a separate ReadMe.md or ReadMe.txt file, or at the top of your notebook.

This analysis asks what portion of jobs within the St. Louis metro area are going to be vulnerable to flooding and climate change? What type of jobs will be vulnerable? The most interesting result from this analysis was 43.3% of transportation and warehousing jobs within the St. Louis Metro area located within levees.

In [1]:
import pandas as pd
import geopandas as gpd

In [ ]:
def function(point, levee):
    #creates geopandas
    point = gpd.read_file(point)
    levee = gpd.read_file(levee)
    #Converts to WGS84
    point = point.to_crs(4326)
    levee = levee.to_crs(4326)
    #spatially joins the points to levees
    join = point.sjoin(levee, how = "left")
    #seperates the data into points inside or outside a levee system
    join_y = join[~join['LEVEED_ID'].isna()]
    join_n = join[join['LEVEED_ID'].isna()]
    #Takes only quant variables, used to create two new dataframes summarizing data
    list = join_y.columns[2:43].tolist()
    data_y = join_y[list].sum()
    data_n = join_n[list].sum()
    #Turns the series into dataframes
    data_y = pd.DataFrame(data_y)
    data_n = pd.DataFrame(data_n)
    #Reorganizes the columns
    data_y = data_y.reset_index().rename(columns={data_y.columns[0]:'data'})
    data_n = data_n.reset_index().rename(columns={data_n.columns[0]:'data'})
    #Merges the two dataframes
    data = pd.merge(data_y, data_n, on = 'index')
    #Renames columns for readability
    data['levee'] = data['data_x']
    data['non_levee'] = data['data_y']
    #Removes unncessary columns
    data = data.drop('data_x', axis=1)
    data = data.drop('data_y', axis=1)
    #Creates a new column which indicates what percent of the category are within a levee
    data['pct_levee'] = data['levee'] / (data['levee'] + data['non_levee'])
    return data

In [39]:
Work_Data = function('STL_Work.json', 'STL_Levee.json')
Home_Data = function('STL_Home.json', 'STL_Levee.json')

In [42]:
Work_Data

,index,levee,non_levee,pct_levee
0,c000,142440.0,1263074.0,0.101344
1,ca01,29406.0,293742.0,0.090999
2,ca02,78151.0,662822.0,0.105471
3,ca03,34883.0,306510.0,0.102178
4,ce01,19772.0,239350.0,0.076304
5,ce02,33819.0,310360.0,0.098260
6,ce03,88849.0,713364.0,0.110755
7,cns01,507.0,3184.0,0.137361
8,cns02,128.0,1138.0,0.101106
9,cns03,712.0,7177.0,0.090252
